# PRISMATIC Workshop: Initializing FATES from NEON Remote Sensing

This notebook walks through the PRISMATIC pipeline — from downloading raw NEON field and airborne data to generating plant functional type (PFT) initial conditions for the FATES dynamic vegetation model.

**Site:** TEAK (Lower Teakettle), California  
**Year:** 2021  
**Model target:** FATES (Functionally Assembled Terrestrial Ecosystem Simulator)

---

## Background

currently a ppt. convert to canva link

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

from hydra import initialize, compose
from omegaconf import OmegaConf

with initialize(config_path='conf', version_base=None):
    cfg = compose(config_name='config')

site = 'TEAK'
year = 2021
site_cfg = cfg.sites[site][year]

print(OmegaConf.to_yaml(site_cfg))

---

## 1. Downloading Field Inventory Data

**Functions:** `download_veg_structure_data`, `download_polygons`, `download_trait_table`

Access NEON's ground-based forest inventory (stem locations, species IDs, DBH) and plot polygon boundaries via the NEON API — the ecological ground truth that anchors everything downstream.

In [ ]:
from initialize.inventory import download_veg_structure_data, download_trait_table
from initialize.plots import download_polygons

# Download stem-level vegetation structure data (species, DBH, height, location)
download_veg_structure_data(cfg, site, year)

# Download NEON plot boundary polygons
download_polygons(cfg, site, year)

# Download NEON trait table (used later for allometric biomass equations)
download_trait_table(cfg)

---

## 2. Downloading Airborne Remote Sensing Data

**Functions:** `download_lidar`, `download_hyperspectral` (or `download_aop_bbox` for a spatial subset)

Retrieve NEON AOP (Airborne Observation Platform) LiDAR point clouds and hyperspectral imagery — the two remote sensing data streams that drive PFT classification at landscape scale.

In [ ]:
from initialize.lidar import download_lidar, download_aop_bbox
from initialize.hyperspectral import download_hyperspectral

# Download LiDAR point clouds (.laz files) for the site/year
# Use download_aop_bbox to restrict to a spatial bounding box if needed
download_lidar(cfg, site, year)

# Download hyperspectral flightline imagery
download_hyperspectral(cfg, site, year)

---

## 3. Processing Field Inventory into Training Labels

**Functions:** `prep_veg_structure`, `prep_polygons`

Clean and filter stem measurements to the target year, assign PFT labels to individual plants, and partition NEON plots into spatial units suitable for linking to remote sensing pixels.

> We will classify PFTs from NEON's airborne LiDAR and hyperspectral data and train the model on ground-based inventories.

In [ ]:
from initialize.inventory import prep_veg_structure
from initialize.plots import prep_polygons

# Filter stems to target year, assign PFT labels
prep_veg_structure(cfg, site, year)

# Partition plot polygons into spatial subunits for RS linking
prep_polygons(cfg, site, year)

---

## 4. Processing LiDAR: Normalization and Clipping

**Functions:** `normalize_laz`, `clip_lidar_by_plots`

Height-normalize raw LiDAR point clouds (removing terrain so heights reflect canopy, not topography), then clip to inventory plot boundaries to align 3D structure data with field observations.

In [ ]:
from initialize.lidar import normalize_laz, clip_lidar_by_plots

# Subtract digital terrain model so z reflects canopy height above ground
normalize_laz(cfg, site, year)

# Clip normalized point clouds to NEON plot boundaries
clip_lidar_by_plots(cfg, site, year)

---

## 5. Deriving Canopy Structure: Leaf Area Density Profiles

**Function:** `prep_lad`

Compute vertical leaf area density (LAD) profiles from normalized LiDAR within each plot, stratified by size class — a key structural descriptor of canopy layering and vegetation density.

### FATES Cohort Structure

The LAD profiles map directly onto FATES cohort height classes. Each cohort in FATES represents plants of similar height competing for light — the canopy layering captured here defines the initial vertical structure of the simulated forest.

![FATES cohort diagram placeholder](docs/fates_cohorts_placeholder.png)

> We will classify PFTs from NEON's airborne LiDAR and hyperspectral data and train the model on ground-based inventories.

In [ ]:
from initialize.lad import prep_lad

# Compute LAD profiles per plot, stratified by PFT size class
prep_lad(cfg, site, year)

---

## 6. Estimating Biomass

**Function:** `prep_biomass`

Apply allometric equations to stem measurements (using a NEON trait table) to estimate above-ground biomass per plant functional type per plot — providing a carbon stock summary alongside structural data.

In [ ]:
from initialize.biomass import prep_biomass

# Estimate AGB per stem using species-specific allometry from the NEON trait table
prep_biomass(cfg, site, year)

---

## 7. Preparing Hyperspectral Imagery

**Functions:** `correct_flightlines`, `prep_aop_imagery`

Apply BRDF and topographic corrections to hyperspectral flightlines, then stack hyperspectral bands with LiDAR-derived rasters into a single multi-layer image ready for pixel-level classification.

> We will classify PFTs from NEON's airborne LiDAR and hyperspectral data and train the model on ground-based inventories.

In [ ]:
from initialize.hyperspectral import prep_aop_imagery
from utils.apply_brdf_corrections import correct_flightlines

# Apply BRDF and topographic corrections across flightlines
correct_flightlines(cfg, site, year)

# Stack corrected hyperspectral bands + LiDAR rasters into a single image
prep_aop_imagery(cfg, site, year)

---

## 8. Building the Training Dataset

**Functions:** `prep_manual_training_data`, `extract_spectra_from_polygon`

Overlay inventory-derived crown polygons onto the stacked AOP imagery to extract per-pixel spectral signatures, producing a labeled training table (spectral features + PFT class) for the classifier.

In [ ]:
from initialize.hyperspectral import extract_spectra_from_polygon

# Extract per-pixel spectra within each crown polygon and attach PFT labels
extract_spectra_from_polygon(cfg, site, year)

---

## 9. Training the PFT Classifier

**Function:** `train_pft_classifier`

Train a Random Forest model on the labeled spectral training data (optionally using PCA instead of raw wavelengths) and evaluate accuracy — producing a model that can predict PFT identity for any pixel in the scene.

In [ ]:
from initialize.hyperspectral import train_pft_classifier

# Train Random Forest on labeled spectral data; evaluate with held-out accuracy
train_pft_classifier(cfg, site, year)

---

## 10. Generating Initial Conditions for FATES

**Function:** `generate_initial_conditions`

Apply the trained classifier wall-to-wall across the site, then aggregate pixel-level PFT predictions and biomass into FATES cohort and patch files — the end product that initializes a land surface model with spatially-informed vegetation structure.

> Here we have now classified plant functional composition (height class, PFT) across a single tile for a NEON site and year.

In [ ]:
from initialize.generate_initial_conditions import generate_initial_conditions

# Classify wall-to-wall, aggregate to cohorts/patches, write FATES IC files
generate_initial_conditions(cfg, site, year)

---

## 11. Where We're Going: FATES Simulations

The initial conditions we just generated feed directly into FATES simulations. Below are example outputs from FATES runs initialized with PRISMATIC data — showing how the spatially-informed PFT structure and biomass translate into simulated forest dynamics over time.

*(Simulation figures go here)*